# Gold 2: icu_clinical_monitoring

One row per ICU stay per day. every event table already
carries stay_id, and the day number is computed from each stay's first recorded
event. Fluid intake comes from ingredientevents (ml only) since raw inputevents
mixes ml with mg, dose and mEq. Config driven like the gold 1.

In [0]:
import json
import uuid
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

with open("/Volumes/cdr_gold/config/gold_config/gold_config.json", "r") as f:
    GCFG = json.load(f)

SILVER = GCFG["sources"]["silver_schema"]
BRONZE = GCFG["sources"]["bronze_schema"]
GOLD   = GCFG["target"]["gold_schema"]
PARAMS = GCFG["parameters"]

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")

def t(name):
    # silversilver tables are pending
    for schema in (SILVER, BRONZE):
        try:
            df = spark.table(f"{schema}.{name}")
            if schema == BRONZE:
                print(f"note: using BRONZE for {name} (silver not found)")
            return df
        except Exception:
            continue
    raise Exception(f"table {name} not found in {SILVER} or {BRONZE}")

def t_any(*names):
    # try several table names (the team silver renames some tables)
    for n in names:
        try:
            return t(n)
        except Exception:
            continue
    raise Exception(f"none of {names} found")

def title_col(df):
    return "description" if "description" in df.columns else "long_title"

def apply_comments(table_key):
    # the Genie semantic layer lives in
    # one reviewable file instead of being scattered through the code
    meta = GCFG["tables"][table_key]
    spark.sql(f"COMMENT ON TABLE {GOLD}.{table_key} IS '{meta['comment']}'")
    for col_name, comment in meta.get("column_comments", {}).items():
        spark.sql(f"COMMENT ON COLUMN {GOLD}.{table_key}.{col_name} IS '{comment}'")
    print(f"comments applied to {table_key}")

def age_group_expr(colname):
    # age buckets built from the configured bounds
    bounds = PARAMS["age_group_bounds"]
    expr, lo = None, 18
    for b in bounds:
        label = f"{lo}-{b-1}"
        expr = F.when(F.col(colname) < b, label) if expr is None else expr.when(F.col(colname) < b, label)
        lo = b
    return expr.otherwise(f"{bounds[-1]}+")

# gold audit
GOLD_AUDIT = f"{GOLD}.gold_audit_log"
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GOLD_AUDIT} (
    run_id           STRING,
    target_table     STRING,
    status           STRING,
    row_count        BIGINT,
    start_time       TIMESTAMP,
    end_time         TIMESTAMP,
    duration_seconds DOUBLE,
    notes            STRING
) USING DELTA
""")

RUN_ID = f"gold_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:6]}"
_build_start = {}

def audit_start(table_key):
    _build_start[table_key] = datetime.now()

def audit_end(table_key, row_count, notes=None):
    end = datetime.now()
    start = _build_start.get(table_key, end)
    dur = (end - start).total_seconds()
    spark.createDataFrame(
        [(RUN_ID, f"{GOLD}.{table_key}", "SUCCESS", row_count, start, end, dur, notes)],
        "run_id string, target_table string, status string, row_count long, "
        "start_time timestamp, end_time timestamp, duration_seconds double, notes string"
    ).write.mode("append").format("delta").saveAsTable(GOLD_AUDIT)
    print(f"audit: {table_key} | {row_count:,} rows | {dur:.1f}s | run {RUN_ID}")

print(f"gold loader ready | schema: {GOLD} | run: {RUN_ID}")

In [0]:
# load the event tables
ie = t("inputevents")
ig = t("ingredientevents")
oe = t("outputevents")
pe = t("procedureevents")
d_items = t("d_items")

# fluid intake per stay per day, from ingredientevents filtered to ml.
def norm_uom(colname):
    return F.lower(F.regexp_replace(F.trim(F.col(colname)), r"\.$", ""))

intake = (ig.filter(norm_uom("amountuom") == "ml")
  .filter(F.col("amount") > 0)
  .withColumn("clinical_date", F.to_date("starttime"))
  .groupBy("stay_id", "clinical_date")
  .agg(F.round(F.sum("amount"), 1).alias("intake_ml")))

# all output and urine specifically 
urine_items = d_items.filter(F.lower("label").rlike(PARAMS["urine_label_regex"])).select("itemid")

out_all = (oe.withColumn("clinical_date", F.to_date("charttime"))
  .groupBy("stay_id", "clinical_date")
  .agg(F.round(F.sum("value"), 1).alias("output_ml")))

out_urine = (oe.join(urine_items, "itemid")
  .withColumn("clinical_date", F.to_date("charttime"))
  .groupBy("stay_id", "clinical_date")
  .agg(F.round(F.sum("value"), 1).alias("urine_output_ml")))